## Creating Spark Session

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

packages = ["org.apache.hadoop:hadoop-aws:3.4.0"]

builder = (
    SparkSession.builder
    .appName("MinIO-Delta-Lake")
    .master("local[*]")
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "miniopass")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=packages
).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 15:40:31 WARN Utils: Your hostname, TIC-06, resolves to a loopback address: 127.0.1.1; using 172.25.124.25 instead (on interface eth0)
26/11/18 15:40:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/realdo/.cache/pypoetry/virtualenvs/projeto-ed-satc-Gz-K1ou1-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/realdo/.ivy2.5.2/cache
The jars for the packages stored in: /home/realdo/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3b8e8a6e-1bf7-4b2d-a276-e44e58934c0e;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4

## Reading Silver Tables

In [2]:
caminho_silver = 's3a://silver'

df_categoria_produto = spark.read.format("delta").load(f"{caminho_silver}/categoria_produto")
df_inpc_valor = spark.read.format("delta").load(f"{caminho_silver}/inpc_valor")
df_ipca_valor = spark.read.format("delta").load(f"{caminho_silver}/ipca_valor")
df_municipio = spark.read.format("delta").load(f"{caminho_silver}/municipio")
df_pib_municipio = spark.read.format("delta").load(f"{caminho_silver}/pib_municipio")
df_pnad_domicilio = spark.read.format("delta").load(f"{caminho_silver}/pnad_domicilio")
df_pnad_morador = spark.read.format("delta").load(f"{caminho_silver}/pnad_morador")
df_pnad_ocupacao = spark.read.format("delta").load(f"{caminho_silver}/pnad_ocupacao")
df_populacao_municipio = spark.read.format("delta").load(f"{caminho_silver}/populacao_municipio")
df_producao_agricola = spark.read.format("delta").load(f"{caminho_silver}/producao_agricola")
df_producao_origem_animal = spark.read.format("delta").load(f"{caminho_silver}/producao_origem_animal")
df_rebanho_municipio = spark.read.format("delta").load(f"{caminho_silver}/rebanho_municipio")

26/11/18 15:40:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [ ]:
def safe_drop(df, cols):
    existing = set(df.columns)
    to_drop = [c for c in cols if c in existing]
    return df.drop(*to_drop) if to_drop else df